# Data Context

This EDA examines the LUND-PROBE dataset (Rogowski et al., 2025), a public benchmark of 432 prostate MRgRT patients. Each patient folder contains a synthetic CT (sCT), an MR image, interpolated dose distributions, and structure segmentations for targets  and organs at risk (OAR). The sCT is used as the primary model input because dose calculation depends on tissue desnity, which MR alone does not provide. 

A secondary metadata CSV ('patGeometryInformation_basePart.csv') and a missing structures JSON file are also available and will be used to assess dataset completeness before any image-level analysis. 

In [9]:
# Standard library
import os
import json

# Third-party: data and visualisation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Medical imaging
import nibabel as nib
from pathlib import Path

# ----------------------------------------------------------------------
# Dataset paths 
# BASE_PATH points to the 432-patient basePart on the network drive.
# The metadata CSV and missing structures JSON sit one level up.
# ----------------------------------------------------------------------
BASE_PATH = Path(r"\\vumc.nl\Onderzoek\s4e-gpfs2\rath-research-01\Research\Research_Kidney_AI_OB_ABR_MP_NGA_Joris\NIKA\LUnd_dataset\lund-probe\lund-probe\basePart")

METADATA_CSV = BASE_PATH.parent / "patGeometryInformation_basePart.csv"
MISSING_JSON = BASE_PATH.parent / "missingStructures_basePart.json"

# Collect patient directories in sorted order for reproducibility
patient_dirs = sorted([d for d in BASE_PATH.iterdir() if d.is_dir()])

print(f"Patients found: {len(patient_dirs)}")
print(f"First patient: {patient_dirs[0].name if patient_dirs else 'None'}")
print(f"Last: {patient_dirs[-1].name if patient_dirs else 'None'}")

Patients found: 432
First patient: newAcq_01d2150e9b50efa1
Last: oldAcq_fffac611ac446b6f


## 1. Corpus-Level Overview

With 432 patients confirmed, the next step is to inspect the metadata files provided with LUND-PROBE. The geometry CSV contains per-patient image dimensions, voxel spacing, and structure volumes. The missing structures JSON flags any patients where segmentations are absent. Together these give a dataset-wide picture before any image-level loading.

In [11]:
# Load the per-patient geometry metadata provided with LUND-PROBE.
# This tells is image dimensions, voxelspacing, intesnity statistics,
# and structure volumes for every patient, without loading a single image.

df = pd.read_csv(METADATA_CSV, sep=";")

print(f"Metadata shape: {df.shape}")  # (rows, columns) — expect ~432 rows
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nFirst two rows:")
df.head(2)

Metadata shape: (432, 9)

Column names:
['patient', 'sCTMatrixSizeOrig', 'sCTVoxelSizeOrig (mm)', 'mriMatrixSize', 'mriVoxelSize (mm)', 'doseMatrixSizeOrig', 'doseVoxelSizeOrig', 'doseMin (Gy)', 'doseMax (Gy) ']

First two rows:


,patient,sCTMatrixSizeOrig,sCTVoxelSizeOrig (mm),mriMatrixSize,mriVoxelSize (mm),doseMatrixSizeOrig,doseVoxelSizeOrig,doseMin (Gy),doseMax (Gy)
0,oldAcq_ca05ccf044225231,"(756, 1024, 88)","(0.4375, 0.4375, 2.5)","(1024, 1024, 88)","(0.4375, 0.4375, 2.5)","(108, 162, 88)","(2.5, 2.5, 2.5)",0.0,44.30
1,oldAcq_0cbde6c35fb03c23,"(756, 1024, 88)","(0.4375, 0.4375, 2.5)","(1024, 1024, 88)","(0.4375, 0.4375, 2.5)","(116, 174, 88)","(2.5, 2.5, 2.5)",0.0,44.19
